# Week 11 — Wednesday: Ensembles

**DATA 202 · Calvin University**

Same pumpkin seeds as Monday. A straight boundary (logistic regression) got 88.9% in cross-validation. Trees can draw bendy boundaries, but one deep tree overfits (Week 7).

> 🎯 **What happens when many trees vote?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~5 min | 1 · One tree, no limit |
| ~10 min | 2 · Five trees vote: random forests (SLO 11B) |
| ~8 min | 3 · Why the forest doesn't overfit (SLO 11B) |
| ~8 min | 4 · Boosting: trees in a row (SLO 11B) |
| ~9 min | 5 · The final comparison |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: Koklu, Sarigil & Ozbek (2021), *Genetic Resources and Crop Evolution* 68(7), 2713–2726</sub>

---
## 1 · One Tree, No Limit · ~5 min

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

seeds = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/pumpkin.csv")
X = seeds.drop(columns="variety")
y = seeds["variety"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)    # Monday's split

tree = DecisionTreeClassifier(random_state=1)
scores = cross_validate(tree, X_train, y_train, cv=5, return_train_score=True)
print(f"one tree, no depth limit | folds it learned from: {scores['train_score'].mean():.3f}"
      f" | held-out fold: {scores['test_score'].mean():.3f}")
tree.fit(X_train, y_train)
print(f"{tree.get_depth()} levels deep, {tree.get_n_leaves()} leaves")

> 💬 **Question:** What is this tree doing? What did we do about it in Week 7?

<details><summary>Answer</summary>

* 100% on the seeds it learned from, 83.0% on held-out seeds, with 195 leaves: it memorized. **Overfitting**
* Week 7's fix: limit the depth. The best depth here is 4, at 87.2%, still below Monday's 88.9%
* Today's fix: keep the deep trees, but grow **many** of them and let them vote
</details>

---
## 2 · Five Trees Vote: Random Forests (SLO 11B) · ~10 min

A **random forest** grows many deep trees, each a little different:

* Each tree learns from its own **bootstrap sample**: 1,875 seeds drawn *with replacement* from the 1,875 training seeds. Some seeds come twice; about a third are left out
* At each split, a tree may only look at a **random few** of the 12 columns
* To predict, the trees **vote**; the majority wins

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/11/images/ensembles.png" width="720" alt="Three diagrams. Bagging: different subsets of the training data each train a base model, and the most frequent or average prediction wins. Boosting: base models in a row, each trained on data modified by the previous model's predictions. Stacking: different models whose predictions feed a meta-learner.">

In [ ]:
forest5 = RandomForestClassifier(n_estimators=5, random_state=1).fit(X_train, y_train)

six = X_test.loc[[2368, 1276, 297, 679, 1561, 655]]
votes = pd.DataFrame(index=["A", "B", "C", "D", "E", "F"])
for i, one_tree in enumerate(forest5.estimators_):
    # the trees inside a forest answer 0 or 1; forest5.classes_ turns that back into names
    votes[f"tree {i + 1}"] = forest5.classes_[one_tree.predict(six.to_numpy()).astype(int)]
votes

> 📄 **Handout 1 — Five trees vote.** Count the votes and fill in the forest's answer. Then check it against the actual variety. 🗣️ Which single tree would you trust alone?

In [ ]:
votes["forest"] = forest5.predict(six)
votes["actual"] = y_test[six.index].to_numpy()
votes

<details><summary>Answer</summary>

* The forest gets **5 of 6** right. Every single tree gets only 3 or 4: each one slips on a different seed, and the others outvote it
* No tree is reliable alone. The **vote** is
* **Seed F** fools all five trees. A vote can't fix a mistake every voter makes
</details>

Does a bigger forest help?

In [ ]:
for n in [1, 5, 10, 100]:
    acc = cross_val_score(RandomForestClassifier(n_estimators=n, random_state=1), X_train, y_train, cv=5).mean()
    print(f"forest of {n:3} trees | 5-fold CV accuracy: {acc:.3f}")

<details><summary>Answer</summary>

* 1 tree: 82.3% · 5 trees: 86.3% · 10 trees: 88.1% · 100 trees (the default): **88.3%**
* Beyond a few dozen trees, the gains stop. More trees never hurt; they just cost time
</details>

---
## 3 · Why the Forest Doesn't Overfit (SLO 11B) · ~8 min

Same knob as Week 7, `max_depth`, for one tree and for a forest of 100. (About 30 seconds to run.)

In [ ]:
rows = []
for depth in [1, 2, 3, 4, 6, 8, 10, 15, None]:
    for name, model in [("one tree", DecisionTreeClassifier(max_depth=depth, random_state=1)),
                        ("forest of 100", RandomForestClassifier(max_depth=depth, random_state=1, n_jobs=-1))]:
        scores = cross_validate(model, X_train, y_train, cv=5, return_train_score=True)
        rows.append({"max_depth": str(depth), "model": name, "scored on": "folds it learned from",
                     "accuracy": scores["train_score"].mean()})
        rows.append({"max_depth": str(depth), "model": name, "scored on": "held-out fold",
                     "accuracy": scores["test_score"].mean()})
fig = px.line(pd.DataFrame(rows), x="max_depth", y="accuracy", color="model", line_dash="scored on", markers=True,
              title="One Tree vs. a Forest, as the Trees Grow Deeper")
fig.show()

> 📄 **Handout 2 — Tree vs. forest.** Label the tree's overfitting side. 🗣️ Both reach 100% on the seeds they learned from. Why doesn't the forest's held-out accuracy fall?

<details><summary>Answer</summary>

* **One tree:** best at depth 4 (87.2%), then down to **83.0%** with no limit: overfitting
* **Forest:** about **88%** from depth 4 on, and still **88.3%** with no limit
* Each deep tree memorizes its own bootstrap sample, with its own quirks. The quirks disagree and get outvoted; the real pattern (long seeds are Ürgüp Sivrisi) is shared, and wins the vote
* Many overfit trees, averaged, overfit much less. So a forest rarely needs a depth limit
</details>

---
## 4 · Boosting: Trees in a Row (SLO 11B) · ~8 min

**Gradient boosting** builds the trees one after another:

* Each new tree is **small** (depth 3 by default) and learns from what the trees before it still **get wrong**
* The trees' answers **add up**. `learning_rate` shrinks each tree's say (default 0.1)
* The knob: `n_estimators`, how many trees in the row

(About 30 seconds to run.)

In [ ]:
rows = []
for n in [1, 3, 10, 30, 100, 300, 1000]:
    scores = cross_validate(GradientBoostingClassifier(n_estimators=n, random_state=1), X_train, y_train,
                            cv=5, return_train_score=True)
    rows.append({"trees": n, "model": "boosting", "scored on": "folds it learned from", "accuracy": scores["train_score"].mean()})
    rows.append({"trees": n, "model": "boosting", "scored on": "held-out fold", "accuracy": scores["test_score"].mean()})
for n in [1, 3, 10, 30, 100, 300]:
    acc = cross_val_score(RandomForestClassifier(n_estimators=n, random_state=1, n_jobs=-1), X_train, y_train, cv=5).mean()
    rows.append({"trees": n, "model": "forest", "scored on": "held-out fold", "accuracy": acc})
fig = px.line(pd.DataFrame(rows), x="trees", y="accuracy", color="model", line_dash="scored on", markers=True, log_x=True,
              title="Adding Trees: Boosting vs. a Forest")
fig.show()

> 📄 **Handout 3 — More trees.** Circle where boosting starts to overfit. 🗣️ Why can more trees hurt boosting but not the forest?

<details><summary>Answer</summary>

* **Boosting:** best around **30 trees** (88.3%). By 1,000 trees it gets 100% of the seeds it learned from and **86.8%** of held-out seeds: **overfitting**
* **Forest:** flat at about 88% once it has 10 or more trees
* A forest's new tree is one more independent vote. A boosting tree chases the mistakes that are left, and in the end those are the odd, noisy seeds: it starts memorizing them
* So boosting's `n_estimators` (with `learning_rate`) must be chosen with cross-validation. Popular versions: XGBoost, LightGBM, CatBoost
</details>

---
## 5 · The Final Comparison · ~9 min

Each model with its knob chosen by cross-validation. Then, once, the locked test set.

In [ ]:
finalists = {
    "logistic regression, C = 100 (Monday)": make_pipeline(StandardScaler(), LogisticRegression(C=100, max_iter=10_000)),
    "one tree, depth 4": DecisionTreeClassifier(max_depth=4, random_state=1),
    "random forest, 100 trees": RandomForestClassifier(random_state=1),
    "boosting, 30 trees": GradientBoostingClassifier(n_estimators=30, random_state=1),
}
rows = []
for name, model in finalists.items():
    cv = cross_val_score(model, X_train, y_train, cv=5).mean()
    test = model.fit(X_train, y_train).score(X_test, y_test)
    rows.append({"model": name, "5-fold CV": round(cv, 3), "test": round(test, 3)})
pd.DataFrame(rows)

> 💬 **Question:** Which model would you ship?

<details><summary>Answer</summary>

* CV ranks logistic regression first (88.9%); the test set ranks the forest first (89.1%). All four land within 2.4 points
* 📌 **One split:** on 625 test seeds, one point is about 6 seeds. The order flips between CV and test, so the differences are mostly luck
* With a tie, prefer the simpler model: Monday's logistic regression, whose slopes we can read
* The forest's real win is over the **single** tree: from 83% back up to 88%, without having to tune a depth
</details>

In [ ]:
wrong = pd.DataFrame({name: model.predict(X_test) != y_test for name, model in finalists.items()})
print(f"test seeds that all four models get wrong: {wrong.all(axis=1).sum()} of {len(y_test)}")

<details><summary>Answer</summary>

* **46** seeds (7%) fool every model: they look like the other variety. That is the ceiling of what shape alone can tell
</details>

> I looked, and there before me was a great multitude that no one could count, from every nation, tribe, people and language, standing before the throne.
> — Revelation 7:9

> 💬 **Question:** A forest beats its trees only when the trees make *different* mistakes, so it builds in diversity on purpose: different samples, different columns. Seed F fooled all five trees. Where have you seen a group whose members all shared the same blind spot? What does a different perspective add that one more of the same cannot?

<details><summary>Answer</summary>

* Five copies of the same tree would vote exactly like one tree. A forest gains nothing from agreement, only from **diverse** members whose errors don't line up
* The flip side: if every member learned from the same skewed data, they share the same blind spot, and the vote keeps it (Week 14, fairness)
* Scripture pictures God's people as one body with many members (1 Cor 12; Rev 5:9; Rev 7:9): the differences are part of the design, not a defect to average away
</details>

---
### Wrap-up

1. One deep tree **overfits**; many deep trees that **vote** (a **random forest**) overfit far less, because their quirks cancel
2. Forests make trees different on purpose: bootstrap samples and random columns. More trees never hurt
3. **Boosting** builds small trees in a row, each fixing the last ones' mistakes. Too many trees **overfits**: choose `n_estimators` with CV
4. On these seeds, every good model lands near 88% 📌. The ensemble beat the single tree, not the simple line